# VAR Backtest — Horizon h = 2

On retire les **2 dernières observations** (janvier et février 2026) du jeu d'entraînement,
on ré-estime le VAR(3), on forecast h=2 avec Monte Carlo, puis on compare aux vraies valeurs.

Variables évaluées : **CPI** et **GDP**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from statsmodels.tsa.api import VAR


## 1. Chargement des données

In [ ]:
df_full = pd.read_csv(r"Book3.csv", dayfirst=True)
variables = ['CPI', 'Sonia', 'Import_CVM', 'Export_cvm', 'GDP', 'wages']

# Valeur wages février (comme dans le modèle original)
df_full.iloc[-1, 6] = 4

df_full = df_full[["date_idx"] + variables].dropna()
print(f"Données complètes : {len(df_full)} lignes")
print(f"Dernières dates : {df_full['date_idx'].tail(4).values}")


## 2. Split train / test

On retire les **2 dernières lignes** = jan 2026 et fév 2026.

In [ ]:
H = 2  # horizon de forecast

df_train = df_full.iloc[:-H].copy()   # tout sauf jan + fév 2026
df_test  = df_full.iloc[-H:].copy()   # jan 2026 et fév 2026 (vraies valeurs)

print("Dernière obs entraînement :", df_train['date_idx'].iloc[-1])
print("Obs à prédire            :", df_test['date_idx'].values)
print()
print("Vraies valeurs CPI  :", df_test['CPI'].values)
print("Vraies valeurs GDP  :", df_test['GDP'].values)


## 3. Log-différenciation et ré-estimation VAR(3)

In [ ]:
non0_variables = ['CPI', 'Sonia', 'Import_CVM', 'Export_cvm', 'GDP']

df_diff_train = pd.DataFrame(columns=variables)
df_diff_train[non0_variables] = np.log(df_train[non0_variables]).diff().dropna()
df_diff_train['wages'] = df_train['wages'].diff()
df_diff_train = df_diff_train.dropna()

chosen_lag = 3
model_bt = VAR(df_diff_train)
fitted_bt = model_bt.fit(chosen_lag)
print(f"VAR({chosen_lag}) estimé sur {len(df_diff_train)} observations (différenciées)")
print(f"Sigma_u shape : {fitted_bt.sigma_u.shape}")


## 4. Ancres de niveaux pour reconstruction

Dans le modèle original :
- `mar25_histo` = `df.iloc[-12]`  →  dans df_train tronqué = `df_train.iloc[-14]`
- `apr25_histo` = `df.iloc[-11]`  →  dans df_train tronqué = `df_train.iloc[-13]`
- `feb_histo`   = `df.iloc[-1]`   →  ici c'est décembre 2025 = `df_train.iloc[-1]`

Le backtest simule la situation où on est **fin décembre 2025** et on forecast jan + fév 2026.

In [ ]:
# Ancres de niveaux (identique à la logique originale, décalées de -H)
jan25_histo = df_train.iloc[-14][variables].values   # janvier 2025  (= mar25 original - 2 mois)
feb25_histo = df_train.iloc[-13][variables].values   # février 2025  (= apr25 original - 2 mois)
dec_histo   = df_train.iloc[-1][variables]           # décembre 2025 (= point de départ forecast)

print("jan25_histo (ancre h=1) :", jan25_histo[[0,4]])  # CPI, GDP
print("feb25_histo (ancre h=2) :", feb25_histo[[0,4]])
print("dec_histo   (dernier obs) :", dec_histo[['CPI','GDP']].values)


## 5. Monte Carlo — N = 10 000 simulations

In [ ]:
rng = np.random.default_rng(seed=42)
i_c = 0   # index CPI
i_g = 4   # index GDP
n_vars = 6
N = 10_000
Sigma_u = fitted_bt.sigma_u

jan_reconst = np.zeros((N, n_vars))
feb_reconst = np.zeros((N, n_vars))

for s in range(N):
    # ── h=1 : forecast janvier 2026 depuis décembre 2025 ──────────────
    point_jan  = fitted_bt.forecast(df_diff_train.values[chosen_lag:], steps=1)  # (1, n_vars)
    noise_jan  = rng.multivariate_normal(np.zeros(n_vars), Sigma_u)
    jan_draw   = point_jan[0] + noise_jan                                         # (n_vars,)
    jan_reconst[s] = dec_histo.values * np.exp(jan_draw)

    # YoY CPI et GDP
    jan_reconst[s][i_c] = jan_reconst[s][i_c] / jan25_histo[i_c] - 1
    jan_reconst[s][i_g] = jan_reconst[s][i_g] / jan25_histo[i_g] - 1

    # ── h=2 : forecast février 2026 depuis décembre + janvier ─────────
    history_jan = np.vstack([df_diff_train.values[chosen_lag:], jan_draw])
    point_feb   = fitted_bt.forecast(history_jan, steps=1)                        # (1, n_vars)
    noise_feb   = rng.multivariate_normal(np.zeros(n_vars), Sigma_u)
    feb_draw    = point_feb[0] + noise_feb
    feb_reconst[s] = jan_reconst[s] * np.exp(feb_draw)

    # YoY CPI et GDP
    feb_reconst[s][i_c] = feb_reconst[s][i_c] / feb25_histo[i_c] - 1
    feb_reconst[s][i_g] = feb_reconst[s][i_g] / feb25_histo[i_g] - 1

# shape : (2, N, n_vars)  — axe 0 = horizon
reconst_bt = np.stack([jan_reconst, feb_reconst])
print("reconst_bt.shape :", reconst_bt.shape)


## 6. Métriques d'erreur

Pour CPI et GDP à h=1 (janvier 2026) et h=2 (février 2026) :
- **Biais** = mean(forecast) − réel
- **MAE** = |mean(forecast) − réel|
- **Couverture** : le vrai tombe-t-il dans l'intervalle à 68 / 90 / 95 % ?

In [ ]:
confidence_levels = {"95": (2.5, 97.5), "90": (5.0, 95.0), "68": (16.0, 84.0)}

eval_vars  = {'CPI': i_c, 'GDP': i_g}
horizons   = {1: ('janvier 2026', jan_reconst), 2: ('février 2026', feb_reconst)}

rows = []
for var_name, idx in eval_vars.items():
    true_val = df_test[var_name].values  # [jan, fev]
    for h_num, (h_label, sims) in enumerate(horizons.items()):
        draws = sims[:, idx]              # (N,)
        mean_fc = np.mean(draws)
        median_fc = np.median(draws)
        bias = mean_fc - true_val[h_num]
        mae  = abs(bias)
        row = {
            'Variable': var_name,
            'Horizon': f'h={h_num+1} ({h_label})',
            'Vrai': round(true_val[h_num], 4),
            'Moyenne MC': round(mean_fc, 4),
            'Médiane MC': round(median_fc, 4),
            'Biais': round(bias, 4),
            'MAE': round(mae, 4),
        }
        for label, (lo, hi) in confidence_levels.items():
            lb = np.percentile(draws, lo)
            ub = np.percentile(draws, hi)
            covered = lb <= true_val[h_num] <= ub
            row[f'Dans IC {label}%'] = '✓' if covered else '✗'
            row[f'IC {label}% bas'] = round(lb, 4)
            row[f'IC {label}% haut'] = round(ub, 4)
        rows.append(row)

df_metrics = pd.DataFrame(rows)
cols_display = ['Variable','Horizon','Vrai','Moyenne MC','Médiane MC',
                'Biais','MAE','Dans IC 68%','Dans IC 90%','Dans IC 95%']
print(df_metrics[cols_display].to_string(index=False))


## 7. Visualisation — distributions MC vs vraie valeur

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Backtest VAR(3) — h=2 | Distributions MC vs vraies valeurs\n'
             'Entraînement : données jusqu\'à déc 2025 | Test : jan–fév 2026',
             fontsize=13, fontweight='bold')

band_colors = {'95': ('#2196F3', 0.12), '90': ('#2196F3', 0.25), '68': ('#2196F3', 0.45)}

plot_specs = [
    (0, 0, 'CPI',  i_c, jan_reconst, 'CPI — h=1 (janvier 2026)'),
    (0, 1, 'CPI',  i_c, feb_reconst, 'CPI — h=2 (février 2026)'),
    (1, 0, 'GDP',  i_g, jan_reconst, 'GDP — h=1 (janvier 2026)'),
    (1, 1, 'GDP',  i_g, feb_reconst, 'GDP — h=2 (février 2026)'),
]

true_vals = {'CPI': df_test['CPI'].values, 'GDP': df_test['GDP'].values}

for r, c, vname, idx, sims, title in plot_specs:
    ax = axes[r, c]
    draws = sims[:, idx]
    h_idx = 0 if 'h=1' in title else 1
    true_v = true_vals[vname][h_idx]

    ax.hist(draws, bins=80, color='#90CAF9', edgecolor='none', alpha=0.7, density=True)

    for label, (lo, hi) in confidence_levels.items():
        lb = np.percentile(draws, lo)
        ub = np.percentile(draws, hi)
        color, alpha = band_colors[label]
        ax.axvspan(lb, ub, color=color, alpha=alpha, label=f'IC {label}%')

    ax.axvline(np.mean(draws), color='#1565C0', lw=2, linestyle='--', label='Moyenne MC')
    ax.axvline(true_v, color='#C62828', lw=2.5, linestyle='-', label=f'Réel = {true_v:.4f}')

    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Valeur YoY')
    ax.set_ylabel('Densité')
    ax.legend(fontsize=8, framealpha=0.8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Cone de prévision — CPI et GDP

In [ ]:
def plot_backtest_cone(var_name, idx, n_hist=24):
    fig, ax = plt.subplots(figsize=(11, 5))

    # Historique (niveaux YoY recalculés)
    hist_vals = df_train[var_name].values
    hist_idx  = list(range(len(hist_vals)))
    ax.plot(hist_idx[-n_hist:], hist_vals[-n_hist:],
            color='#212121', lw=2, label='Observé (train)')

    # Vraies valeurs de test
    test_x = [len(hist_vals), len(hist_vals)+1]
    test_y = df_test[var_name].values
    ax.plot(test_x, test_y, color='#C62828', lw=2, marker='o',
            linestyle='-', label='Réel (test jan–fév 2026)')

    # Forecast MC
    means = [np.mean(jan_reconst[:, idx]), np.mean(feb_reconst[:, idx])]
    ax.plot([len(hist_vals)-1, len(hist_vals), len(hist_vals)+1],
            [hist_vals[-1]] + means,
            color='#1565C0', lw=2, linestyle='--', label='Forecast MC (moyenne)')

    # Bandes
    for label, (lo, hi) in confidence_levels.items():
        color, alpha = band_colors[label]
        lbs = [np.percentile(jan_reconst[:, idx], lo),
               np.percentile(feb_reconst[:, idx], lo)]
        ubs = [np.percentile(jan_reconst[:, idx], hi),
               np.percentile(feb_reconst[:, idx], hi)]
        x_band = [len(hist_vals)-1, len(hist_vals), len(hist_vals)+1]
        ax.fill_between(x_band,
                        [hist_vals[-1]] + lbs,
                        [hist_vals[-1]] + ubs,
                        color=color, alpha=alpha+0.05, label=f'IC {label}%')

    ax.axvline(len(hist_vals)-0.5, color='grey', lw=1, linestyle=':', alpha=0.8)
    ax.set_title(f'{var_name} — Backtest h=2 | Train: jusqu\'à déc 2025',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Indice temporel')
    ax.set_ylabel(f'{var_name} YoY')
    ax.legend(fontsize=9, framealpha=0.8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_backtest_cone('CPI', i_c)
plot_backtest_cone('GDP', i_g)


## 9. Tableau de synthèse complet

In [ ]:
# Tableau avec intervalles détaillés
print('\n' + '='*80)
print('RÉSULTATS DU BACKTEST — VAR(3) | h=2')
print('='*80)
for _, row in df_metrics.iterrows():
    print(f"\n{row['Variable']} | {row['Horizon']}")
    print(f"  Vraie valeur   : {row['Vrai']:.4f}")
    print(f"  Moyenne MC     : {row['Moyenne MC']:.4f}")
    print(f"  Biais          : {row['Biais']:+.4f}")
    print(f"  MAE            : {row['MAE']:.4f}")
    for label in ['68', '90', '95']:
        lb = row[f'IC {label}% bas']
        ub = row[f'IC {label}% haut']
        ok = row[f'Dans IC {label}%']
        print(f"  IC {label}%       : [{lb:.4f}, {ub:.4f}]  {ok}")
print('\n' + '='*80)
